# Photon Mosaic vs. CaImAn: same ROIs, same movie

Do photon-mosaic's `regression` fluorescence extraction and CNMF `background` give the same
traces as CaImAn's CNMF? This notebook

1. generates the same synthetic movie as `play_with_analyzer.ipynb`,
2. runs CaImAn's `CNMF.fit` seeded with the true ROIs (no detection, `p=0`) -- in a separate
   process, since CaImAn is not a photon-mosaic dependency (`run_caiman_seeded.py`),
3. hands CaImAn's estimated footprints `A` to photon-mosaic and extracts traces with
   `method="regression"` against a CNMF `background`,
4. compares the two, on the raw traces, on the background, and on ΔF/F.

Needs a Python environment with CaImAn; point `CAIMAN_PYTHON` at it (below).

In [ ]:
import hashlib
import os
import subprocess
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
from ipywidgets import Dropdown, interactive_output

import photon_mosaic as pm
from photon_mosaic.core.numpyimaging import NumpyRois

%matplotlib widget

# Python interpreter that has CaImAn installed; adjust, or set the CAIMAN_PYTHON environment variable.
CAIMAN_PYTHON = os.environ.get("CAIMAN_PYTHON", str(Path.home() / "miniconda3/envs/caiman/bin/python"))
WORKDIR = Path(tempfile.gettempdir()) / "photon_mosaic_vs_caiman"
WORKDIR.mkdir(exist_ok=True)

## Generate synthetic imaging data

Identical to `play_with_analyzer.ipynb`, including the neuropil under the ROIs being 0.7x the neuropil around them.

In [ ]:
# Background under the ROIs relative to around them (cell bodies displace neuropil); 0.7 is also what
# Suite2p's default neuropil_weight assumes. photon-mosaic's corrections below get the same value.
NEUROPIL_COEFFICIENT = 0.7

rois, imaging, ground_truth = pm.generate_imaging_with_rois(
    num_frames=10000,
    bleaching_time=600.0,
    noise_std="poisson",
    weighted_rois=True,
    neuropil_model="vignette",  # one shared fluctuation x a static vignetting profile
    neuropil_roi_coefficient=NEUROPIL_COEFFICIENT,
    seed=0,
)
t = np.arange(imaging.get_num_samples()) / imaging.sampling_frequency
num_rois = rois.get_num_rois()

## CaImAn's CNMF, seeded with the true ROIs

`run_caiman_seeded.py` builds `CNMF(Ain=<true masks>, p=0)` and calls `fit`, with no patches
(`rf=None`) and no merging, so ROI `k` stays ROI `k`. Three things that are not obvious:

- **The seed has to be boolean.** For a float `Ain`, `fit` raises ("You need to provide estimate of
  `C` and `f`"); for a boolean one CaImAn derives its own initial `C`, `b`, `f` from the mask. So
  the true masks' weights are dropped: only each ROI's support is used.
- **`fit` still refines `A`.** The seed fixes the initial support, but `fit` runs a spatial and a
  temporal update, then a second refinement round, so the `A` that comes out is not the seed.
- **`p=0`** skips the AR deconvolution: `C` is the demixed fluorescence (non-negative), not a
  denoised trace, which is what photon-mosaic's `fluorescence` extraction produces too.

In [ ]:
movie = imaging.get_series()[..., 0].astype(np.float32)  # (T, H, W): single plane
cache_key = hashlib.md5(movie[::50].tobytes()).hexdigest()[:8]  # ties the cached CaImAn result to this data
movie_path, masks_path = WORKDIR / f"movie_{cache_key}.npy", WORKDIR / f"true_masks_{cache_key}.npy"
caiman_path = WORKDIR / f"caiman_result_{cache_key}.npz"

if not caiman_path.exists():  # cached: delete the file to re-run CaImAn
    np.save(movie_path, movie)
    np.save(masks_path, np.asarray(rois.get_roi_image_masks()))
    run = subprocess.run(
        [CAIMAN_PYTHON, "run_caiman_seeded.py", "--movie", str(movie_path), "--masks", str(masks_path),
         "--out", str(caiman_path), "--fs", str(imaging.sampling_frequency), "--p", "0", "--gnb", "1"],
        capture_output=True, text=True,
    )
    if run.returncode:
        raise RuntimeError(run.stdout[-2000:] + run.stderr[-2000:])

caiman = np.load(caiman_path)
height, width = caiman["dims"]
# CaImAn flattens images in Fortran order; its A has unit-norm columns, C is in matching units.
caiman_masks = np.stack([caiman["A"][:, k].reshape(height, width, order="F") for k in range(num_rois)])
caiman_trace = (caiman["C"] + caiman["YrA"]).T  # demixed trace, (T, K); with p=0 this is C plus its residual
caiman_f = caiman["f"].T  # (T, 1): the single background component's time course
true_masks = np.asarray(rois.get_roi_image_masks())
mask_corr = [np.corrcoef(true_masks[k].ravel(), caiman_masks[k].ravel())[0, 1] for k in range(num_rois)]
print(f"true mask vs CaImAn's refined A, per-ROI correlation: median {np.median(mask_corr):.3f}, min {np.min(mask_corr):.3f}")
print("caiman A (support px):", (caiman_masks > 0).sum(axis=(1, 2))[:6], "... vs true:",
      (true_masks > 0).sum(axis=(1, 2))[:6], "...")

## photon-mosaic on CaImAn's ROIs

CaImAn's footprints become a photon-mosaic ROI set, so both pipelines use *the same* `A`. Then
`background` (CNMF, `gnb=1` -- vignette is rank 1) and `fluorescence` with `method="regression"`,
the joint solve over all ROIs and the background. Because CaImAn's `A` columns have unit norm,
photon-mosaic's traces come out in the same units as CaImAn's `C`. `neuropil_weight` is how much of
the fitted background is assumed to sit under the ROIs: 1 keeps the fit's own split, and a smaller
value (here the true 0.7) returns `1 - w` of it to the traces. CaImAn has no such parameter.

In [ ]:
caiman_rois = NumpyRois(caiman_masks, sampling_frequency=imaging.sampling_frequency)
analyzer = pm.create_roi_analyzer(caiman_rois, imaging)
background_ext = analyzer.compute("background", gnb=1, n_jobs=4)
fluorescence_kwargs = dict(method="regression", n_jobs=4, use_neuropil=True, neuropil_source="background")

# neuropil_weight=1 (the default) keeps the joint fit's own split of trace and background ...
analyzer.compute("fluorescence", **fluorescence_kwargs)
dff_pm_joint = analyzer.compute("df_over_f", n_jobs=4, method="percentile").get_data()

# ... while a weight below 1 leaves (1 - w) of the fitted background in the traces, as surround's
# F - w * Fneu does. Matching the neuropil that is really under the ROIs is the best choice.
fluorescence_ext = analyzer.compute("fluorescence", neuropil_weight=NEUROPIL_COEFFICIENT, **fluorescence_kwargs)
dff_ext = analyzer.compute("df_over_f", n_jobs=4, method="percentile")
pm_trace = fluorescence_ext.get_data()

## Do the traces agree?

Raw, per ROI: correlation and the slope of CaImAn's trace on photon-mosaic's. Then the same after
projecting out a constant and CaImAn's background time course `f`. That second comparison is the
meaningful one: the CNMF model `Y ≈ C Aᵀ + f bᵀ` has an exact gauge freedom, `b → b + A α`,
`C → C − f αᵀ`, which leaves the fit unchanged but moves a trace's offset and `f`-shaped
component between the trace and the background (see `NeuropilExtension`). Each implementation
settles on its own member of that family, so only the part orthogonal to `f` and the constant is
determined.

In [ ]:
def corr_cols(a, b):
    a, b = a - a.mean(0), b - b.mean(0)
    return (a * b).sum(0) / np.sqrt((a * a).sum(0) * (b * b).sum(0))


def project_out(x, regressors):
    coef, *_ = np.linalg.lstsq(regressors, x, rcond=None)
    return x - regressors @ coef


gauge = np.hstack([np.ones((len(t), 1)), caiman_f])  # [1, f]: the directions the fit cannot pin down
pm_proj, caiman_proj = project_out(pm_trace, gauge), project_out(caiman_trace, gauge)

corr_raw = corr_cols(pm_trace, caiman_trace)
corr_proj = corr_cols(pm_proj, caiman_proj)
slope_proj = (pm_proj * caiman_proj).sum(0) / (pm_proj**2).sum(0)

print(f"{'':>26}  {'median':>8}  {'min':>8}")
print(f"{'corr, raw traces':>26}  {np.median(corr_raw):>8.4f}  {corr_raw.min():>8.4f}")
print(f"{'corr, [1, f] projected out':>26}  {np.median(corr_proj):>8.4f}  {corr_proj.min():>8.4f}")
print(f"{'slope, [1, f] projected out':>26}  {np.median(slope_proj):>8.4f}  {slope_proj.min():>8.4f}")

fig_trace, ax_trace = plt.subplots(2, 1, figsize=(10, 3.5), sharex=True)


def _plot_traces(roi):
    for a in ax_trace:
        a.clear()
    ax_trace[0].plot(t, pm_trace[:, roi], lw=0.5, label="photon-mosaic")
    ax_trace[0].plot(t, caiman_trace[:, roi], lw=0.5, label="CaImAn")
    ax_trace[0].set_ylabel("raw trace")
    ax_trace[0].set_title(f"ROI {roi}: corr {corr_raw[roi]:.3f} raw, {corr_proj[roi]:.3f} with [1, f] projected out", fontsize=9)
    ax_trace[1].plot(t, pm_proj[:, roi], lw=0.5, label="photon-mosaic")
    ax_trace[1].plot(t, caiman_proj[:, roi], lw=0.5, label="CaImAn")
    ax_trace[1].set_ylabel("[1, f] projected out")
    ax_trace[1].set_xlabel("Time (s)")
    ax_trace[1].set_xlim(t[0], t[-1])
    ax_trace[0].legend(loc="upper right", fontsize=8)
    fig_trace.tight_layout()


roi_dropdown = Dropdown(options=list(range(num_rois)), value=8, description="ROI:")
display(roi_dropdown, interactive_output(_plot_traces, {"roi": roi_dropdown}))

## The background

The two backgrounds have the same time course (up to scale) but differ in how much of each ROI's
level they claim. The ground truth tells which split is closer to the truth: the neuropil level each
one removes from an ROI, against `ground_truth.neuropil` (scale `1.0` = unbiased). For photon-mosaic
that is `neuropil_weight` times `B`; CaImAn has no such weight and removes its whole background.

In [ ]:
def fit_scale(inferred, truth):
    return (inferred * truth).sum(axis=0) / (inferred * inferred).sum(axis=0)


A, b_caiman = caiman["A"], caiman["b"][:, 0]
to_pixel_units = (A**2).sum(0) / A.sum(0)  # same conversion as play_with_analyzer.ipynb
neuropil_pm = fluorescence_ext.get_data(key="background") * to_pixel_units
neuropil_caiman = caiman_f * (A * b_caiman[:, None]).sum(0) / A.sum(0)  # f(t) * mean of b under each footprint

print(f"{'':>26}  {'scale':>7}  {'corr':>7}")
for name, est in (
    ("photon-mosaic, weight 1", neuropil_pm),
    (f"photon-mosaic, weight {NEUROPIL_COEFFICIENT}", NEUROPIL_COEFFICIENT * neuropil_pm),
    ("CaImAn", neuropil_caiman),
):
    print(f"{name:>26}  {np.median(fit_scale(est, ground_truth.neuropil)):>7.3f}  "
          f"{np.median(corr_cols(est, ground_truth.neuropil)):>7.4f}")

b_pm = background_ext.get_data()[0, :, :, 0]
fig_b, ax_b = plt.subplots(1, 2, figsize=(9, 4))
vmax = np.percentile(np.concatenate([b_pm.ravel(), b_caiman]), 99.5)
for a, img, title in zip(ax_b, (b_pm, b_caiman.reshape(height, width, order="F")), ("photon-mosaic b", "CaImAn b")):
    im = a.imshow(img, vmin=0, vmax=vmax)
    a.set_title(title)
    a.axis("off")
fig_b.colorbar(im, ax=ax_b, shrink=0.8)

## ΔF/F

CaImAn's `detrend_df_f` and photon-mosaic's `df_over_f` both fold the background into the
denominator, `(C − C₀) / (B₀ + C₀)`, which is meant to make ΔF/F robust to exactly that gauge
freedom. Same 60 s window and automatic percentile on both sides. photon-mosaic is shown with the
joint fit's own split (`neuropil_weight=1`) and with the true weight; the last row runs it from the
*true* masks instead of CaImAn's refined ones.

In [ ]:
analyzer_true = pm.create_roi_analyzer(rois, imaging)
analyzer_true.compute("background", gnb=1, n_jobs=4)
analyzer_true.compute(
    "fluorescence",
    method="regression",
    n_jobs=4,
    use_neuropil=True,
    neuropil_source="background",
    neuropil_weight=NEUROPIL_COEFFICIENT,
)
dff_true_masks = analyzer_true.compute("df_over_f", n_jobs=4, method="percentile").get_data()

dff_pm, dff_caiman = dff_ext.get_data(), caiman["F_dff"].T
truth = ground_truth.clean_traces
for name, dff in (("weight 1", dff_pm_joint), (f"weight {NEUROPIL_COEFFICIENT}", dff_pm)):
    print(
        f"median corr, photon-mosaic ({name}) vs CaImAn dF/F: {np.median(corr_cols(dff, dff_caiman)):.4f}, "
        f"[1, f] projected out of both: "
        f"{np.median(corr_cols(project_out(dff, gauge), project_out(dff_caiman, gauge))):.4f}"
    )
print()

truth_proj = project_out(truth, gauge)
print(f"{'median corr with ground truth':>44}  {'dF/F':>7}  {'[1, f] projected out':>21}")
for name, dff in (
    ("photon-mosaic (CaImAn's A), w=1", dff_pm_joint),
    (f"photon-mosaic (CaImAn's A), w={NEUROPIL_COEFFICIENT}", dff_pm),
    ("CaImAn", dff_caiman),
    (f"photon-mosaic (true masks), w={NEUROPIL_COEFFICIENT}", dff_true_masks),
):
    print(f"{name:>44}  {np.median(corr_cols(dff, truth)):>7.4f}  "
          f"{np.median(corr_cols(project_out(dff, gauge), truth_proj)):>21.4f}")

fig_dff, ax_dff = plt.subplots(figsize=(10, 1.8))


def _plot_dff(roi):
    ax_dff.clear()
    ax_dff.plot(t, 100 * dff_pm[:, roi], lw=0.5, alpha=0.7, label="photon-mosaic")
    ax_dff.plot(t, 100 * dff_caiman[:, roi], lw=0.5, alpha=0.7, label="CaImAn")
    ax_dff.plot(t, 100 * truth[:, roi], lw=0.5, ls="-.", c="k", label="ground truth")
    ax_dff.set_ylabel(r"$\Delta F/F$ [%]")
    ax_dff.set_xlabel("Time (s)")
    ax_dff.set_xlim(t[0], t[-1])
    ax_dff.legend(loc="upper right", fontsize=8)
    fig_dff.tight_layout()


display(roi_dropdown, interactive_output(_plot_dff, {"roi": roi_dropdown}))

## Should they agree?

Partly, and the notebook shows which part.

- **The determined part of the traces agrees almost exactly.** With the same `A` (and so the same
  units), the traces match with median correlation 0.999 and slope 0.99 once the constant and the
  `f`-shaped component are projected out. The raw traces correlate at only 0.62, because the model
  can't say how much of that component belongs to the trace and how much to the background, and
  the two implementations make different choices.
- **The backgrounds differ in level, not in shape.** Both follow the true neuropil time course
  (correlation 0.9998 and 0.997). On this data, where the neuropil under the ROIs is 0.7x the
  neuropil around them, the joint fit's own split (`neuropil_weight=1`) removes about 1.6x too much
  (scale 0.65), and CaImAn about 3.5x too much (scale 0.29): its background has taken over most of
  each ROI's baseline. photon-mosaic fills the background in across each footprint from the
  surrounding pixels before fitting (`_inpaint_roi_support`), which corresponds to weight 1;
  `neuropil_weight=0.7` then corrects it (scale 0.92). CaImAn has no counterpart.
- **ΔF/F follows the background.** Against the ground truth, photon-mosaic reaches 0.80 with weight 1
  and 0.84 with the true weight, CaImAn 0.53. The weight moves photon-mosaic's ΔF/F towards the truth
  and away from CaImAn's: it agrees with CaImAn at 0.85 with weight 1 and 0.69 with weight 0.7.

Caveats: one synthetic dataset and seed; the weight is set to the true coefficient, which real data
doesn't reveal; CaImAn is run with its settings left at defaults apart from `p=0`, `gnb=1`, no merging,
a boolean seed and no patches; and the ground truth is built to match photon-mosaic's assumption of a
background that is smooth across the ROIs. Treat the ground-truth rows as a statement about this data,
not about the two packages in general.